# 04: MARE (Memory-Augmented Recommendation Engine) LTR Model
**AI Shopping OS — Memory-Augmented Multi-Agent AI Shopping Assistant**

This notebook trains an **XGBoost LambdaMART (Ranker)** learning-to-rank model using an 8-factor feature representation:
1. `preference` (brand/category alignment)
2. `price` (discount + budget fit)
3. `review` (rating + sentiment)
4. `spec` (feature quality tags)
5. `twinCompat` (behavioral match)
6. `memory` (explicit memory signals)
7. `context` (intent match)
8. `availability` (stock + delivery SLA)


In [ ]:
# Step 1: Install Dependencies
!pip install -q xgboost lightgbm scikit-learn torch pandas numpy


In [ ]:
# Step 2: Generate Ranking Dataset (Queries & Candidates)
import numpy as np
import pandas as pd
import xgboost as xgb

np.random.seed(42)
NUM_QUERIES = 200
ITEMS_PER_QUERY = 10

feature_names = ["preference", "price", "review", "spec", "twinCompat", "memory", "context", "availability"]

X_rows = []
y_relevance = []
query_groups = []

for qid in range(NUM_QUERIES):
    # Dynamic category weights for ground truth calculation
    category_weights = np.random.dirichlet(np.ones(8))
    
    for item_idx in range(ITEMS_PER_QUERY):
        feats = np.random.uniform(20, 100, size=8)
        # Compute ground truth score 0..4
        raw_score = np.dot(feats, category_weights)
        rel_label = int(np.clip((raw_score - 30) / 15, 0, 4))
        
        X_rows.append(feats)
        y_relevance.append(rel_label)
    
    query_groups.append(ITEMS_PER_QUERY)

X = pd.DataFrame(X_rows, columns=feature_names)
y = np.array(y_relevance)

# Train / Test split
split_q = 160 * ITEMS_PER_QUERY
X_train, X_test = X.iloc[:split_q], X.iloc[split_q:]
y_train, y_test = y[:split_q], y[split_q:]
groups_train = query_groups[:160]
groups_test = query_groups[160:]

print(f"Created LTR dataset with {NUM_QUERIES} query groups, {len(X)} total candidate rows.")


In [ ]:
# Step 3: Train XGBoost LambdaMART Ranker
ranker = xgb.XGBRanker(
    objective="rank:ndcg",
    eval_metric=["ndcg@6", "mrr@6"],
    n_estimators=100,
    learning_rate=0.08,
    max_depth=4,
    random_state=42
)

ranker.fit(
    X_train, y_train,
    group=groups_train,
    eval_set=[(X_test, y_test)],
    eval_group=[groups_test],
    verbose=True
)


In [ ]:
# Step 4: Evaluate Feature Importance
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
importance = ranker.feature_importances_
plt.barh(feature_names, importance, color='teal')
plt.title("MARE Score Factor Feature Importance (LambdaMART)")
plt.xlabel("Importance Score")
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig("mare_feature_importance.png")
plt.show()


In [ ]:
# Step 5: Export Model
ranker.save_model("mare_ranker_lambdamart.json")
print("Saved MARE LambdaMART model to mare_ranker_lambdamart.json")
